In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.process_tree.obj import ProcessTree
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.objects.petri_net.exporter import exporter as pnml_exporter

# INDUCTIVE MINER

xes_file = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_script.xes"
output_png = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_inductive.png"

log = xes_importer.apply(xes_file)
print("Log Imported")

res = inductive_miner.apply(log, variant=inductive_miner.Variants.IM)

if isinstance(res, ProcessTree):
    net, initial_marking, final_marking = pt_converter.apply(
        res, variant=pt_converter.Variants.TO_PETRI_NET
    )
else:
    net, initial_marking, final_marking = res

print("Model discovered")

#output_pnml = r"C:\Users\lomo0\Downloads\900Less88\VM_pn_mined3.pnml"
#pnml_exporter.apply(net, initial_marking, output_pnml, final_marking=final_marking)

parameters = {
    pn_visualizer.Variants.FREQUENCY.value.Parameters.FORMAT: "png"
}

gviz = pn_visualizer.apply(net, initial_marking, final_marking, parameters=parameters)
pn_visualizer.save(gviz, output_png)
print(f"Petri net saved as PNG")



parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
Model discovered
Petri net saved as PNG


In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.visualization.petri_net import visualizer as pn_visualizer

# HEURISTICS MINER

xes_file = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_script.xes"
output_png = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_heuristic.png"

# Load the log
log = xes_importer.apply(xes_file)
print("Log Imported")

# Apply Heuristic Miner
parameters = {
    heuristics_miner.Variants.CLASSIC.value.Parameters.DEPENDENCY_THRESH: 0.95,
    heuristics_miner.Variants.CLASSIC.value.Parameters.AND_MEASURE_THRESH: 0.9,
    heuristics_miner.Variants.CLASSIC.value.Parameters.LOOP_LENGTH_TWO_THRESH: 0.95,
    heuristics_miner.Variants.CLASSIC.value.Parameters.MIN_ACT_COUNT: 3,
    heuristics_miner.Variants.CLASSIC.value.Parameters.MIN_DFG_OCCURRENCES: 4,
}


net, initial_marking, final_marking = heuristics_miner.apply(log, parameters=parameters)

print("Heuristic miner model discovered")

# Visualize and save as PNG
gviz = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(gviz, output_png)
print(f"Petri net saved to {output_png}")


parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
Heuristic miner model discovered
Petri net saved to C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_heuristic.png


In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.process_tree.obj import ProcessTree
from pm4py.visualization.petri_net import visualizer as pn_visualizer

from pm4py.objects.petri_net.exporter import exporter as pnml_exporter

# INDUCTIVE INFREQUENT MINER

xes_file = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_script.xes"
output_png = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_imf3_09.png"

log = xes_importer.apply(xes_file)
print("Log Imported")

parameters = {"noiseThreshold": 0.9} 
res = inductive_miner.apply(log, variant=inductive_miner.Variants.IMf, parameters=parameters)
print("IMf miner model discovered")

if isinstance(res, ProcessTree):
    net, initial_marking, final_marking = pt_converter.apply(
        res, variant=pt_converter.Variants.TO_PETRI_NET
    )
else:
    net, initial_marking, final_marking = res

print("Model discovered")

#output_pnml = r"C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_imf3.pnml"
#pnml_exporter.apply(net, initial_marking, output_pnml, final_marking=final_marking)


gviz = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(gviz, output_png)
print(f"Petri net saved as PNG to {output_png}")


parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
IMf miner model discovered
Model discovered
Petri net saved as PNG to C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_imf3_09.png


In [ ]:
import xml.etree.ElementTree as ET

# XES CLEANING (for Los Alamos logs)

INPUT_XES = "C:/Users/lomo0/Downloads/wls_processtype.xes"
OUTPUT_XES = "C:/Users/lomo0/Downloads/wls_processtype_cleaned.xes"

# Load XES
tree = ET.parse(INPUT_XES)
root = tree.getroot()

ns = {"xes": "http://www.xes-standard.org/"}

def is_login(event_elem):
    concept_name_elem = event_elem.find("xes:string[@key='concept:name']", ns)
    if concept_name_elem is not None:
        return concept_name_elem.attrib["value"].startswith("4624")
    return False

traces_to_remove = []

for trace in root.findall("xes:trace", ns):
    events = trace.findall("xes:event", ns)
    login_found = False
    new_events = []

    for event in events:
        if not login_found and is_login(event):
            login_found = True 
        if login_found:
            new_events.append(event)

    for e in events:
        trace.remove(e)

    for e in new_events:
        trace.append(e)

    if not new_events:
        traces_to_remove.append(trace)

for trace in traces_to_remove:
    root.remove(trace)

print(f"Removed {len(traces_to_remove)} traces with no login events")
print(f"Remaining traces: {len(root.findall('xes:trace', ns))}")

# Prettify
def indent(elem, level=0):
    i = "\n" + level*"  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            indent(child, level+1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if not elem.tail or not elem.tail.strip():
            elem.tail = i

indent(root)
ET.register_namespace('', "http://www.xes-standard.org/")  # <--- add this
tree.write(OUTPUT_XES, encoding="UTF-8", xml_declaration=True)

print(f"✅ Cleaned XES written to {OUTPUT_XES}")


Removed 536 traces with no login events
Remaining traces: 60829
✅ Cleaned XES written to C:/Users/lomo0/Downloads/wls_processtype_cleaned.xes
